# Student Performance Prediction System — Week 1
## Data Understanding & Analysis

**Workflow covered this week:** Data → Cleaning → Analysis → Feature Selection

**Week 1 checklist**
1. Import libraries
2. Load the dataset
3. Understand columns & data types
4. Data cleaning (duplicates, inconsistent text, invalid values)
5. Handle missing values
6. Exploratory Data Analysis (EDA)
7. Visualizations
8. Patterns & relationships
9. Select important features for model training

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression

sns.set_theme(style="whitegrid")
os.makedirs("visualizations", exist_ok=True)
os.makedirs("data", exist_ok=True)

## 2. Load the Dataset
The dataset is a **synthetic dataset** created with `generate_dataset.py` (1,000 students + a few deliberately
duplicated rows). It mimics real school data and contains realistic data-quality problems
(missing values, duplicates, invalid entries, inconsistent text) so we can practise cleaning.

In [ ]:
df = pd.read_csv("data/student_performance.csv")
df.head()

## 3. Understand the Dataset

In [ ]:
print("Shape (rows, columns):", df.shape)
df.info()

In [ ]:
df.describe().T

In [ ]:
df.describe(include='object').T

### Column dictionary

| Column | Type | Description |
|---|---|---|
| Student_ID | Text | Unique ID of the student |
| Gender | Categorical | Male / Female |
| Age | Numeric | Age in years (15–19) |
| Study_Hours_Per_Week | Numeric | Hours the student studies weekly |
| Attendance_Percentage | Numeric | % of classes attended (0–100) |
| Previous_Exam_Score | Numeric | Score in the previous exam (0–100) |
| Assignments_Completed_Pct | Numeric | % of assignments submitted |
| Sleep_Hours | Numeric | Average sleep per night |
| Internet_Access | Categorical | Yes / No |
| Parental_Education | Categorical | High School / Bachelor / Master / PhD |
| Extracurricular | Categorical | Takes part in activities? Yes / No |
| Tutoring | Categorical | Has extra tutoring? Yes / No |
| **Final_Score** | Numeric | **Target (regression)** – final exam score |
| **Performance** | Categorical | **Target (classification)** – Low (<=50), Average (50–70), High (>70) |

We can use either `Final_Score` (regression) or `Performance` (classification) as the target in Week 2.

## 4. Data Cleaning

### 4.1 Check for problems first

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nGender values:", df['Gender'].unique())
print("\nAttendance > 100:", (df['Attendance_Percentage'] > 100).sum())
print("Study hours > 60/week:", (df['Study_Hours_Per_Week'] > 60).sum())

### 4.2 Remove duplicate rows

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows. New shape: {df.shape}")

### 4.3 Fix inconsistent text (e.g. `male`, `FEMALE`, `'Male '`)

In [ ]:
df['Gender'] = df['Gender'].str.strip().str.title()
print(df['Gender'].value_counts())

### 4.4 Fix invalid values
- Attendance cannot be more than 100 % → treat as missing.
- Nobody can realistically study more than 60 hours a week (a week has 168 h and school/sleep take most of it) → treat as missing.

We set them to `NaN` and fill them together with the other missing values in the next step.

In [ ]:
df.loc[df['Attendance_Percentage'] > 100, 'Attendance_Percentage'] = np.nan
df.loc[df['Study_Hours_Per_Week'] > 60, 'Study_Hours_Per_Week'] = np.nan
print(df[['Attendance_Percentage', 'Study_Hours_Per_Week']].describe().T[['min', 'max']])

## 5. Handle Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing': missing, 'Percent': missing_pct})[missing > 0]

In [ ]:
plt.figure(figsize=(9, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing Values Map (yellow = missing)")
plt.tight_layout()
plt.savefig("visualizations/01_missing_values.png", dpi=150)
plt.show()

**Strategy**
- **Numeric columns** → fill with the **median** (robust to outliers).
- **Categorical columns** → fill with the **mode** (most frequent value).
- Missing percentages are all below ~10 %, so filling is better than throwing away rows.

In [ ]:
num_cols = ['Study_Hours_Per_Week', 'Attendance_Percentage', 'Previous_Exam_Score',
            'Assignments_Completed_Pct', 'Sleep_Hours']
cat_cols_fill = ['Parental_Education', 'Internet_Access']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
for col in cat_cols_fill:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())

### Final cleaned data check

In [ ]:
df.info()

## 6. Exploratory Data Analysis (EDA)

### 6.1 Target distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['Final_Score'], bins=30, kde=True, ax=ax[0], color="steelblue")
ax[0].set_title("Distribution of Final Score")

order = ['Low', 'Average', 'High']
sns.countplot(x='Performance', data=df, order=order, ax=ax[1], hue="Performance", palette="Set2", legend=False)
ax[1].set_title("Performance Categories")
plt.tight_layout()
plt.savefig("visualizations/02_target_distribution.png", dpi=150)
plt.show()

print(df['Performance'].value_counts(normalize=True).round(3) * 100)

### 6.2 Distribution of numeric features

In [ ]:
num_features = ['Age', 'Study_Hours_Per_Week', 'Attendance_Percentage', 'Previous_Exam_Score',
                'Assignments_Completed_Pct', 'Sleep_Hours']
df[num_features].hist(figsize=(13, 7), bins=25, color="teal", edgecolor="white")
plt.suptitle("Distribution of Numeric Features", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig("visualizations/03_numeric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.3 Outlier check (box plots)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, col in zip(axes.ravel(), num_features):
    sns.boxplot(y=df[col], ax=ax, color="lightcoral")
    ax.set_title(col)
plt.tight_layout()
plt.savefig("visualizations/04_boxplots.png", dpi=150)
plt.show()

### 6.4 Categorical features vs. performance

In [ ]:
cat_features = ['Gender', 'Internet_Access', 'Parental_Education', 'Extracurricular', 'Tutoring']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), cat_features):
    sns.countplot(x=col, hue='Performance', data=df, hue_order=order, ax=ax, palette="Set2")
    ax.set_title(f"{col} vs Performance")
    ax.tick_params(axis='x', rotation=15)
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.savefig("visualizations/05_categorical_vs_performance.png", dpi=150)
plt.show()

In [ ]:
# Average final score for each category value
for col in cat_features:
    print(df.groupby(col)['Final_Score'].mean().round(2).sort_values(ascending=False), "\n")

## 7. Relationships Between Variables

### 7.1 Scatter plots: key numeric features vs. Final Score

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.ravel(), ['Study_Hours_Per_Week', 'Attendance_Percentage',
                                  'Previous_Exam_Score', 'Assignments_Completed_Pct']):
    sns.regplot(x=col, y='Final_Score', data=df, ax=ax,
                scatter_kws={'alpha': 0.35, 's': 18}, line_kws={'color': 'red'})
    ax.set_title(f"{col} vs Final Score")
plt.tight_layout()
plt.savefig("visualizations/06_scatter_vs_final_score.png", dpi=150)
plt.show()

### 7.2 Correlation heatmap

In [ ]:
corr = df[num_features + ['Final_Score']].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("visualizations/07_correlation_heatmap.png", dpi=150)
plt.show()

corr['Final_Score'].drop('Final_Score').sort_values(ascending=False)

### 7.3 Feature averages by performance group

In [ ]:
group_means = df.groupby('Performance')[num_features].mean().loc[order].round(2)
group_means

In [ ]:
group_means.T.plot(kind='bar', figsize=(11, 5), colormap="viridis")
plt.title("Average Feature Values by Performance Group")
plt.ylabel("Average value")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig("visualizations/08_group_means.png", dpi=150)
plt.show()

## 8. Key Insights / Patterns Found

1. **Study hours per week** (r ≈ 0.55) and **previous exam score** (r ≈ 0.51) have the strongest positive relationship with the final score.
2. **Attendance** (r ≈ 0.31) and **assignment completion** (r ≈ 0.23) show moderate positive relationships.
3. **Sleep hours** is weakly related (r ≈ 0.10) and **age** has no relationship (r ≈ 0.00).
4. Students with **tutoring** score about 4–5 points higher on average, and those with **internet access** about 3 points higher.
5. Students whose parents only finished **high school** have the lowest average score; the other groups are similar.
6. **Gender and extracurricular activity** make no meaningful difference.
7. The classes are **imbalanced**: Average ≈ 60 %, Low ≈ 27 %, High ≈ 13 %. In Week 2, use a *stratified* train/test split if predicting `Performance`.

## 9. Feature Selection

We use **three views** and combine them:
1. **Correlation** with the target (linear relationship).
2. **Mutual information** (captures non-linear relationships).
3. **Random Forest feature importance**.

Categorical columns are converted to numbers first (encoding).

In [ ]:
# Prepare model-ready copy: drop ID and the two targets, encode categoricals
X = df.drop(columns=['Student_ID', 'Final_Score', 'Performance']).copy()
y = df['Final_Score']

# Binary columns -> 0/1
for col in ['Internet_Access', 'Extracurricular', 'Tutoring']:
    X[col] = X[col].map({'Yes': 1, 'No': 0})
X['Gender'] = X['Gender'].map({'Male': 1, 'Female': 0})
# Ordered category -> ranks
X['Parental_Education'] = X['Parental_Education'].map(
    {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3})

X.head()

In [ ]:
corr_scores = X.corrwith(y).abs()
mi_scores = pd.Series(mutual_info_regression(X, y, random_state=42), index=X.columns)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(X, y)
rf_scores = pd.Series(rf.feature_importances_, index=X.columns)

# Scale each to 0-1 so they can be compared, then average
def norm(s): return (s - s.min()) / (s.max() - s.min())
summary = pd.DataFrame({
    'Correlation': corr_scores, 'Mutual_Info': mi_scores, 'RF_Importance': rf_scores
})
summary['Average_Rank_Score'] = summary.apply(norm).mean(axis=1)
summary = summary.sort_values('Average_Rank_Score', ascending=False).round(3)
summary

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x=summary['Average_Rank_Score'], y=summary.index, hue=summary.index, palette="crest", legend=False)
plt.title("Feature Importance (combined score)")
plt.xlabel("Combined importance (0-1)")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("visualizations/09_feature_importance.png", dpi=150)
plt.show()

### Final feature selection

In [ ]:
THRESHOLD = 0.10
selected_features = summary[summary['Average_Rank_Score'] >= THRESHOLD].index.tolist()
dropped_features = [c for c in X.columns if c not in selected_features]
print("Selected features:", selected_features)
print("Dropped features :", dropped_features)

**Decision:** We keep the features with a combined score above the threshold, because they carry real
predictive signal, and drop the rest (e.g. age, gender, extracurricular) to keep the model simple and reduce noise.
`Student_ID` is dropped because it is only an identifier.

## 10. Save Cleaned Data for Week 2

In [ ]:
cleaned = df.copy()
cleaned.to_csv("data/cleaned_student_data.csv", index=False)

model_ready = X[selected_features].copy()
model_ready['Final_Score'] = y
model_ready['Performance'] = df['Performance']
model_ready.to_csv("data/model_ready_data.csv", index=False)

print("Saved: data/cleaned_student_data.csv", cleaned.shape)
print("Saved: data/model_ready_data.csv    ", model_ready.shape)

## Week 1 Summary
- Loaded and understood a 1,000-student dataset with 14 columns.
- Removed 15 duplicates, standardised text, handled invalid values and filled all missing values.
- Produced 9 visualizations (saved in `visualizations/`).
- Identified the strongest drivers of performance and selected the final features.
- Saved `cleaned_student_data.csv` and `model_ready_data.csv`, ready for **Week 2: model training**.